# Targeting Rule v1

Builds a first, simple rule for whom to email, to answer challenge questions 3 and 4:

3. If only 10,000 customers can receive an email, which ones should?
4. If 10,000 customers must be left out, which ones should be?

The rule ranks customer cells by the extra spend per customer that the better email brings in. All choices below were fixed in [`DECISIONS.md`](../DECISIONS.md) (2026-11-06 and 2026-11-07) before any cell-level effect was estimated. Stage 04 compares uplift models with this rule on the same test set.

## Step 0: Build the customer cells

Each cell crosses past-year spend with past purchases:

- **Past-year spend:** `history_level` from the segment analysis, with levels 3 to 5 merged into "$200 or more". The merge keeps at least 1,000 customers in each email group of each cell in the training data.
- **Past purchases:** men's only, women's only, or both.

The data's `segment` column is the email group each customer was assigned to, not a customer segment. The new column is therefore called `cell`.

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("../data/hillstrom.csv")

LEVEL_LABELS = {1: "under $100", 2: "$100-$200", 3: "$200 or more"}
df["level"] = df["history_segment"].str[0].astype(int).clip(upper=3)
df["bought"] = np.select(
    [(df["mens"] == 1) & (df["womens"] == 1), df["mens"] == 1],
    ["both", "men's only"],
    default="women's only",
)
df["cell"] = df["level"].map(LEVEL_LABELS) + ", " + df["bought"]
CELL_ORDER = df.sort_values(["level", "bought"])["cell"].unique().tolist()

counts = pd.crosstab([df["level"], df["bought"]], df["segment"], margins=True).rename(index=LEVEL_LABELS, level=0)
counts.index.names, counts.columns.name = ["Past-year spend", "Past purchases"], None
counts

Mens E-Mail  No E-Mail  Womens E-Mail    All
Past-year spend Past purchases                                              
under $100      men's only             3889       3829           3804  11522
                women's only           3835       3783           3830  11448
$100-$200       men's only             2296       2464           2358   7118
                women's only           2395       2372           2369   7136
$200 or more    both                   2181       2149           2118   6448
                men's only             3373       3345           3460  10178
                women's only           3338       3364           3448  10150
All                                   21307      21306          21387  64000

In [2]:
# Expected size of the smallest email group in the training half.
group_sizes = pd.crosstab(df["cell"], df["segment"])
print(f"Cells: {df['cell'].nunique()}")
print(f"Smallest email group in a cell: {group_sizes.values.min():,} customers, "
      f"about {group_sizes.values.min() // 2:,} in the training half")

Cells: 7
Smallest email group in a cell: 2,118 customers, about 1,059 in the training half


In [3]:
# Past-year spend of customers who bought both lines, and of everyone under $200.
both = df[df["bought"] == "both"]
print(f"Bought both: {len(both):,} customers, lowest past-year spend ${both['history'].min():.2f} "
      f"({(both['history'] == both['history'].min()).sum()} customer)")
print(f"Bought both, $200-$210: {both['history'].between(200, 210).sum()}; $210-$220: {both['history'].between(210, 220).sum()}")
print(f"Under $200: {(df['history'] < 200).sum():,} customers, of whom {(both['history'] < 200).sum()} bought both")

Bought both: 6,448 customers, lowest past-year spend $200.13 (1 customer)
Bought both, $200-$210: 203; $210-$220: 229
Under $200: 37,224 customers, of whom 0 bought both


**Reading the result:** the merge gives **seven cells, and every email group in every cell has at least 2,118 customers,** or about 1,059 in the training half, just above the minimum of 1,000. The split in step 1 is random, so the actual training counts are checked again there.

**Unexplained pattern at $200:** no customer with past-year spend under $200 bought both lines. Among the 6,448 who did, the lowest past-year spend is $200.13, held by one customer, and 203 customers fall between $200 and $210, about as many as between $210 and $220. Unlike the $499 pattern in spend, no single amount stands out; the group simply starts at $200. A boundary this sharp is unlikely to come from customer behavior alone and may reflect how the dataset was built, but **the cause is unknown**. It is recorded as an anomaly in the [EDA summary](../reports/00_01_eda_summary.md) and under challenge question 7 in the README. It does not change the cells, because there is simply no "both" cell below $200.

## Step 1: Split the data

A 50:50 split, stratified by assigned email group, with seed 42. The rule is built on the training half and evaluated on the test half. A 50:50 split is used instead of the usual 80:20, because the test half must estimate the email effect among the selected customers, which needs more customers than checking prediction accuracy (see `DECISIONS.md`, 2026-11-07). Stage 04 reuses the same split.

In [4]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.5, stratify=df["segment"], random_state=42)

train_sizes = pd.crosstab(train["cell"], train["segment"]).loc[CELL_ORDER]
print(f"Training: {len(train):,} customers; test: {len(test):,} customers")
print(f"Smallest email group in a training cell: {train_sizes.values.min():,} customers (minimum 1,000)")
train_sizes

Training: 32,000 customers; test: 32,000 customers
Smallest email group in a training cell: 1,063 customers (minimum 1,000)


segment,Mens E-Mail,No E-Mail,Womens E-Mail
cell,,,
"under $100, men's only",1975,1936,1885
"under $100, women's only",1912,1926,1886
"$100-$200, men's only",1140,1202,1188
"$100-$200, women's only",1182,1187,1223
"$200 or more, both",1095,1073,1063
"$200 or more, men's only",1666,1663,1766
"$200 or more, women's only",1683,1666,1683


**Reading the result:** the split gives 32,000 customers in each half, and **the smallest email group in a training cell has 1,063 customers, so every cell meets the minimum of 1,000.**

## Step 2: Estimate each email's effect in each cell (training half)

For each cell, one regression `spend ~ mens_email + womens_email` with robust (HC3) standard errors gives both effects against No E-Mail at once. Each cell then receives the email with the higher point estimate, as fixed in `DECISIONS.md`.

The chosen email's effect is classified with the amended rule (`DECISIONS.md`, 2026-11-06). Break-even is the cost per email divided by the gross margin, $0.008 / 0.40 = $0.02 per customer:

- **Worth sending:** the lower end of the 95% confidence interval is above $0.02.
- **Uncertain:** the estimate is above $0.02, but the interval includes it.
- **Not worth sending:** the estimate is at or below $0.02.

In [5]:
import statsmodels.formula.api as smf

MARGIN, COST = 0.40, 0.008
BREAK_EVEN = COST / MARGIN

train = train.assign(mens_email=(train["segment"] == "Mens E-Mail").astype(int),
                     womens_email=(train["segment"] == "Womens E-Mail").astype(int))

rows = []
for cell, data in train.groupby("cell"):
    fit = smf.ols("spend ~ mens_email + womens_email", data).fit(cov_type="HC3")
    ci = fit.conf_int()
    row = {"cell": cell, "customers": len(data)}
    for email, column in [("Mens", "mens_email"), ("Womens", "womens_email")]:
        row[f"{email}_effect"] = fit.params[column]
        row[f"{email}_low"], row[f"{email}_high"] = ci.loc[column]
    rows.append(row)
effects = pd.DataFrame(rows).set_index("cell").loc[CELL_ORDER]


def classify(estimate, low):
    if low > BREAK_EVEN:
        return "worth sending"
    if estimate > BREAK_EVEN:
        return "uncertain"
    return "not worth sending"


effects["chosen_email"] = np.where(effects["Mens_effect"] >= effects["Womens_effect"], "Mens", "Womens")
effects["chosen_effect"] = np.where(effects["chosen_email"] == "Mens", effects["Mens_effect"], effects["Womens_effect"])
effects["chosen_low"] = np.where(effects["chosen_email"] == "Mens", effects["Mens_low"], effects["Womens_low"])
effects["chosen_high"] = np.where(effects["chosen_email"] == "Mens", effects["Mens_high"], effects["Womens_high"])
effects["profit_per_customer"] = MARGIN * effects["chosen_effect"] - COST
effects["category"] = [classify(est, low) for est, low in zip(effects["chosen_effect"], effects["chosen_low"])]

display(effects[["customers", "Mens_effect", "Mens_low", "Mens_high", "Womens_effect", "Womens_low", "Womens_high"]].round(2))
effects[["chosen_email", "chosen_effect", "chosen_low", "chosen_high", "profit_per_customer", "category"]].round(3)

,customers,Mens_effect,Mens_low,Mens_high,Womens_effect,Womens_low,Womens_high
cell,,,,,,,
"under $100, men's only",5796,0.84,-0.22,1.90,0.19,-0.66,1.04
"under $100, women's only",5724,0.34,-0.46,1.15,0.70,-0.16,1.55
"$100-$200, men's only",3530,1.47,0.23,2.71,0.97,0.06,1.88
"$100-$200, women's only",3592,0.42,-0.61,1.46,0.38,-0.54,1.30
"$200 or more, both",3231,2.27,0.00,4.54,-0.31,-1.68,1.06
"$200 or more, men's only",5095,0.11,-0.78,1.01,-0.47,-1.24,0.30
"$200 or more, women's only",5032,1.56,0.56,2.56,0.73,0.04,1.42


,chosen_email,chosen_effect,chosen_low,chosen_high,profit_per_customer,category
cell,,,,,,
"under $100, men's only",Mens,0.840,-0.218,1.899,0.328,uncertain
"under $100, women's only",Womens,0.698,-0.157,1.554,0.271,uncertain
"$100-$200, men's only",Mens,1.474,0.235,2.713,0.582,worth sending
"$100-$200, women's only",Mens,0.424,-0.609,1.456,0.161,uncertain
"$200 or more, both",Mens,2.270,0.002,4.539,0.900,uncertain
"$200 or more, men's only",Mens,0.114,-0.779,1.008,0.038,uncertain
"$200 or more, women's only",Mens,1.561,0.564,2.559,0.617,worth sending


**Reading the result:** **six of the seven cells receive the Mens email.** Only "under \$100, women's only" receives the Womens email, which is consistent with the Mens email's overall lead in stage 01 and with the Womens email working mainly for women's buyers in stage 02.

**Two cells are worth sending and five are uncertain; none is classified as not worth sending.** The worth-sending cells are "\$100-\$200, men's only" and "\$200 or more, women's only", 17,268 customers in the full data. "\$200 or more, both" has the largest estimate, +\$2.27, but its interval runs from \$0.002 to \$4.54. The lower end falls just below the \$0.02 break-even, so the cell is uncertain under the rule fixed on 2026-11-06. Its interval is the widest, because it is the smallest cell and its customers' spend varies most.

**The estimates are noisy, so the ranking is a best bet, not a finding.** Most intervals are about \$2 wide, and neighboring cells can differ more than seems plausible: the Mens email adds \$1.47 for men's-only buyers at \$100-\$200 but only \$0.11 at \$200 or more. The cell ranked first is also the one most likely to be overstated, because the largest of several noisy estimates tends to be too high. The test half checks this in step 5.

## Steps 3 and 4: Rank the cells and choose 10,000 customers

Cells are ranked by the chosen email's estimated effect in dollars per customer, as fixed in `DECISIONS.md` (2026-11-03). The category is shown beside the rank but does not change it (`DECISIONS.md`, 2026-11-09).

- **Question 3 (send to 10,000):** take whole cells from the top of the ranking until the next cell would exceed 10,000, then fill the rest with a random sample from that cell.
- **Question 4 (suppress 10,000):** the same procedure from the bottom of the ranking.

The rule treats every customer in a cell the same, so the sample within the boundary cell is random, with seed 42. Both selections are made from all 64,000 customers; step 5 evaluates them on the customers who fall in the test half.

In [6]:
N_TARGET = 10_000
SEED = 42

ranking = effects[["chosen_email", "chosen_effect", "chosen_low", "category"]].sort_values("chosen_effect", ascending=False)
ranking.insert(0, "rank", range(1, len(ranking) + 1))
ranking["customers"] = df["cell"].value_counts()
ranking["cumulative"] = ranking["customers"].cumsum()


def select(cells, n=N_TARGET, seed=SEED):
    """Take whole cells in the given order, then a random sample of the boundary cell."""
    chosen, remaining = [], n
    for cell in cells:
        members = df[df["cell"] == cell]
        if len(members) <= remaining:
            chosen.append(members.index)
            remaining -= len(members)
        else:
            chosen.append(members.sample(remaining, random_state=seed).index)
            break
    return chosen[0].append(chosen[1:])


df["send_top"] = df.index.isin(select(ranking.index))
df["suppress"] = df.index.isin(select(ranking.index[::-1]))

ranking["send_top"] = df.groupby("cell")["send_top"].sum()
ranking["suppress"] = df.groupby("cell")["suppress"].sum()
print(f"Question 3: {df['send_top'].sum():,} customers selected; question 4: {df['suppress'].sum():,} suppressed; "
      f"overlap: {(df['send_top'] & df['suppress']).sum()}")
ranking.round(2)

Question 3: 10,000 customers selected; question 4: 10,000 suppressed; overlap: 0


,rank,chosen_email,chosen_effect,chosen_low,category,customers,cumulative,send_top,suppress
cell,,,,,,,,,
"$200 or more, both",1,Mens,2.27,0.00,uncertain,6448,6448,6448,0
"$200 or more, women's only",2,Mens,1.56,0.56,worth sending,10150,16598,3552,0
"$100-$200, men's only",3,Mens,1.47,0.23,worth sending,7118,23716,0,0
"under $100, men's only",4,Mens,0.84,-0.22,uncertain,11522,35238,0,0
"under $100, women's only",5,Womens,0.70,-0.16,uncertain,11448,46686,0,0
"$100-$200, women's only",6,Mens,0.42,-0.61,uncertain,7136,53822,0,0
"$200 or more, men's only",7,Mens,0.11,-0.78,uncertain,10178,64000,0,10000


**Reading the result:** **question 3's 10,000 customers are all 6,448 in "\$200 or more, both" plus a random 3,552 of the 10,150 in "\$200 or more, women's only", all sent the Mens email.** Only the second cell is worth sending; the first is ranked first on its point estimate but is uncertain.

**Question 4's 10,000 suppressed customers are a random 10,000 of the 10,178 in "\$200 or more, men's only",** where the Mens email's estimated effect is only \$0.11. These customers spent heavily in the past year, so suppressing them would look wrong if customers were ranked by how much they spend rather than by how much the email changes their spend. This cell's estimate is also one of the noisiest, so step 5 checks both selections on the test half before either is recommended. The two selections do not overlap.